# 🔬 QUBO — Quadratic Unconstrained Binary Optimization

> *"QUBO is the universal language for combinatorial optimization on quantum hardware."*

**What you will gain from this notebook:**

- **QUBO framework** — formulate optimization problems as $\mathbf{x}^T Q\,\mathbf{x}$ over binary vectors
- **QUBO $\leftrightarrow$ Ising equivalence** — transform between binary and spin representations
- **Problem encodings** — map Number Partitioning, Vertex Cover, and Graph Coloring to QUBO
- **Penalty methods** — enforce constraints via quadratic penalty functions
- **QAOA integration** — solve QUBO problems end-to-end with variational quantum circuits

---
## Setup & Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.colors import Normalize as mplNormalize

from qiskit import QuantumCircuit, transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.quantum_info import Statevector, Operator
from scipy.optimize import minimize

backend = BasicSimulator()
SHOTS = 4096
SEED = 42
np.random.seed(SEED)

# ======== Helper functions ================================================

def qubo_cost(x, Q):
    """Compute x^T Q x for a binary vector x."""
    x = np.asarray(x, dtype=float)
    return float(x @ Q @ x)

def brute_force_qubo(Q):
    """Enumerate all 2^n binary vectors; return (best_x, best_cost, all_results)."""
    n = Q.shape[0]
    best_cost, best_x = np.inf, None
    all_results = []
    for k in range(2**n):
        x = np.array([(k >> i) & 1 for i in range(n)], dtype=float)
        c = qubo_cost(x, Q)
        all_results.append((k, x.copy(), c))
        if c < best_cost:
            best_cost, best_x = c, x.copy()
    return best_x, best_cost, all_results

def plot_qubo_landscape(Q, title="QUBO Cost Landscape", ax=None):
    """Bar chart of all 2^n costs; optimal solutions highlighted in green."""
    n = Q.shape[0]
    _, best_cost, all_results = brute_force_qubo(Q)
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(8, 2**n * 0.6), 4))
    labels, costs = [], []
    for k, x, c in all_results:
        labels.append("".join(str(int(b)) for b in x))
        costs.append(c)
    colors = ["#4CAF50" if abs(c - best_cost) < 1e-9 else "#2196F3" for c in costs]
    ax.bar(range(len(costs)), costs, color=colors, edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(len(costs)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=8)
    ax.set_ylabel("Cost f(x)")
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    return ax

print("\u2705 Setup complete \u2014 backend:", backend.name)

---
## Part I — The QUBO Framework

### Definition

**Quadratic Unconstrained Binary Optimization (QUBO):**

$$\min_{\mathbf{x} \in \{0,1\}^n} f(\mathbf{x}) = \mathbf{x}^T Q\,\mathbf{x} = \sum_{i=1}^{n} \sum_{j=1}^{n} Q_{ij}\, x_i\, x_j$$

where $Q$ is an $n \times n$ real matrix (typically **upper-triangular** or symmetric).

**Key properties:**

| Component | Role |
|-----------|------|
| **Diagonal** $Q_{ii}$ | Linear cost for variable $x_i$ (since $x_i^2 = x_i$ for binary) |
| **Off-diagonal** $Q_{ij}$ ($i < j$) | Quadratic interaction between $x_i$ and $x_j$ |

Because $x_i \in \{0,1\}$ implies $x_i^2 = x_i$, the full expansion is:

$$f(\mathbf{x}) = \sum_i Q_{ii}\, x_i \;+\; \sum_{i < j} Q_{ij}\, x_i\, x_j$$

Linear and quadratic terms mix naturally in the diagonal of $Q$.

In [ ]:
# ======== A simple 3-variable QUBO ========================================

Q3 = np.array([
    [-5,  2,  4],
    [ 0, -3,  1],
    [ 0,  0, -8]
], dtype=float)

print("Q =")
print(Q3)

best_x3, best_cost3, all3 = brute_force_qubo(Q3)

print("\nCost landscape:")
print(f"  {'x':<10s}  {'f(x)':<8s}")
print("  " + "-" * 22)
for k, x, c in all3:
    bits = "".join(str(int(b)) for b in x)
    marker = "  \u2705" if abs(c - best_cost3) < 1e-9 else ""
    print(f"  {bits:<10s}  {c:8.1f}{marker}")

print(f"\nOptimal: x* = [{', '.join(str(int(b)) for b in best_x3)}]  f(x*) = {best_cost3}")

fig, ax = plt.subplots(figsize=(8, 4))
plot_qubo_landscape(Q3, title="3-Variable QUBO Landscape", ax=ax)
plt.tight_layout()
plt.show()

assert abs(best_cost3 - (-10.0)) < 1e-9, f"Expected min cost -10, got {best_cost3}"
assert list(best_x3.astype(int)) == [0, 1, 1], f"Expected x*=[0,1,1], got {best_x3}"
print("\u2705 3-variable QUBO validated")

### Why QUBO matters

- **NP-hard** in general — no known polynomial-time algorithm.
- **Universal reduction target**: many combinatorial problems (MaxCut, TSP, graph coloring, scheduling) reduce to QUBO in polynomial time.
- **Natural formulation for quantum optimization**: QAOA and quantum annealing both operate on Ising / QUBO cost functions.

---
## Part II — QUBO $\leftrightarrow$ Ising Equivalence

### The Ising model

The classical Ising Hamiltonian with local fields:

$$H(\mathbf{s}) = \sum_{i<j} J_{ij}\, s_i\, s_j \;+\; \sum_i h_i\, s_i, \qquad s_i \in \{-1,+1\}$$

### Variable substitution

$$x_i = \frac{1 - s_i}{2} \qquad \Longleftrightarrow \qquad s_i = 1 - 2\,x_i$$

### From binary to quantum

Replace each spin variable $s_i$ with the Pauli-$Z$ operator:

$$Z_i\,|0\rangle = +|0\rangle, \qquad Z_i\,|1\rangle = -|1\rangle$$

The Ising Hamiltonian becomes a **diagonal quantum operator**:

$$\hat{H} = \sum_{i<j} J_{ij}\, Z_i Z_j \;+\; \sum_i h_i\, Z_i \;+\; \text{offset} \cdot I$$

**Minimizing the QUBO $=$ finding the ground state of $\hat{H}$.**

In [ ]:
# ======== QUBO -> Ising conversion ========================================

def qubo_to_ising(Q):
    """Convert QUBO matrix Q to Ising parameters (J, h, offset).

    Returns (J, h, offset) such that for s_i = 1 - 2*x_i:
        x^T Q x = sum_{i<j} J[i,j]*s_i*s_j + sum_i h[i]*s_i + offset
    """
    n = Q.shape[0]
    Qs = (Q + Q.T) / 2.0
    J = np.zeros((n, n))
    h = np.zeros(n)
    offset = 0.0
    for i in range(n):
        offset += Qs[i, i] / 2.0
        h[i] -= Qs[i, i] / 2.0
        for j in range(i + 1, n):
            J[i, j] = Qs[i, j] / 2.0
            offset += Qs[i, j] / 2.0
            h[i] -= Qs[i, j] / 2.0
            h[j] -= Qs[i, j] / 2.0
    return J, h, offset

# Convert the 3-variable QUBO from Part I
J3, h3, offset3 = qubo_to_ising(Q3)
n3 = Q3.shape[0]

print("Ising parameters for the 3-variable QUBO:")
print(f"  J = ")
print(J3)
print(f"  h = {h3}")
print(f"  offset = {offset3}")

# Verify: Ising energy must equal QUBO cost for every assignment
print("\nVerification (all 8 assignments):")
for k in range(2**n3):
    x = np.array([(k >> i) & 1 for i in range(n3)], dtype=float)
    s = 1.0 - 2.0 * x
    qubo_val = qubo_cost(x, Q3)
    ising_val = offset3
    for i in range(n3):
        ising_val += h3[i] * s[i]
        for j in range(i + 1, n3):
            ising_val += J3[i, j] * s[i] * s[j]
    bits = "".join(str(int(b)) for b in x)
    print(f"  x={bits}: QUBO={qubo_val:7.1f}  Ising={ising_val:7.1f}")
    assert abs(qubo_val - ising_val) < 1e-9, f"Mismatch at x={bits}!"

print("\n\u2705 QUBO \u2192 Ising conversion verified")

In [ ]:
# ======== Ising -> QUBO conversion ========================================

def ising_to_qubo(J, h):
    """Convert Ising (J, h) to upper-triangular QUBO matrix.

    Returns (Q, offset) such that:
        sum_{i<j} J[i,j]*s_i*s_j + sum_i h[i]*s_i = x^T Q x + offset
    """
    n = len(h)
    Q = np.zeros((n, n))
    offset = 0.0
    for i in range(n):
        Q[i, i] = -2.0 * h[i]
        offset += h[i]
    for i in range(n):
        for j in range(i + 1, n):
            Q[i, j] = 4.0 * J[i, j]
            Q[i, i] -= 2.0 * J[i, j]
            Q[j, j] -= 2.0 * J[i, j]
            offset += J[i, j]
    return Q, offset

# Round-trip: QUBO -> Ising -> QUBO
Q_rt, offset_rt = ising_to_qubo(J3, h3)
shift = offset3 + offset_rt

print("Round-trip test: QUBO \u2192 Ising \u2192 QUBO")
print(f"  Original  Q diagonal: {np.diag(Q3)}")
print(f"  Recovered Q diagonal: {np.diag(Q_rt)}")
print(f"  Constant shift: {shift}")

for k in range(2**n3):
    x = np.array([(k >> i) & 1 for i in range(n3)], dtype=float)
    orig = qubo_cost(x, Q3)
    recov = qubo_cost(x, Q_rt) + shift
    assert abs(orig - recov) < 1e-9, f"Round-trip mismatch at k={k}"

print("\n\u2705 Round-trip QUBO \u2192 Ising \u2192 QUBO preserves all costs")

In [ ]:
# ======== Build the quantum Hamiltonian ===================================

def ising_to_hamiltonian(J, h, n):
    """Diagonal of the Ising Hamiltonian (length 2^n).

    H = sum_{i<j} J[i,j] Z_i Z_j + sum_i h[i] Z_i
    """
    dim = 2**n
    diag = np.zeros(dim)
    for k in range(dim):
        s = np.array([1 - 2 * ((k >> i) & 1) for i in range(n)], dtype=float)
        for i in range(n):
            diag[k] += h[i] * s[i]
            for j in range(i + 1, n):
                diag[k] += J[i, j] * s[i] * s[j]
    return diag

diag3 = ising_to_hamiltonian(J3, h3, n3)

print("Hamiltonian eigenvalues vs QUBO costs:")
for k in range(2**n3):
    x = np.array([(k >> i) & 1 for i in range(n3)], dtype=float)
    qubo_val = qubo_cost(x, Q3)
    bits = "".join(str(int(b)) for b in x)
    print(f"  |{bits}\u27e9  eig={diag3[k]:8.2f}  +offset={diag3[k]+offset3:8.2f}  QUBO={qubo_val:7.1f}")
    assert abs(qubo_val - (diag3[k] + offset3)) < 1e-9

# Cross-check with explicit tensor-product construction
I2 = np.eye(2)
Z = np.array([[1, 0], [0, -1]], dtype=float)

def kron_op(op, qubit, n):
    """I \u2297 ... \u2297 op \u2297 ... \u2297 I  with op on given qubit."""
    result = np.array([[1.0]])
    for q in range(n - 1, -1, -1):
        result = np.kron(result, op if q == qubit else I2)
    return result

H_full = np.zeros((2**n3, 2**n3))
for i in range(n3):
    H_full += h3[i] * kron_op(Z, i, n3)
    for j in range(i + 1, n3):
        H_full += J3[i, j] * (kron_op(Z, i, n3) @ kron_op(Z, j, n3))

assert np.allclose(np.diag(H_full), diag3), "Diagonal mismatch!"
assert np.allclose(H_full - np.diag(diag3), 0), "Hamiltonian must be diagonal!"

print("\n\u2705 Ising \u2192 Hamiltonian verified (diagonal, eigenvalues = Ising energies)")

### Equivalence chain

$$\boxed{\text{QUBO}\; \mathbf{x}^T Q\,\mathbf{x}} \;\;\overset{x_i=(1-s_i)/2}{\longleftrightarrow}\;\; \boxed{\text{Ising}\; \sum J_{ij}s_is_j + \sum h_i s_i} \;\;\overset{s_i \to Z_i}{\longleftrightarrow}\;\; \boxed{\text{Quantum}\; \hat{H}}$$

| Representation | Variables | Objective | Quantum connection |
|----------------|-----------|-----------|-------------------|
| **QUBO** | $x_i \in \{0,1\}$ | $\mathbf{x}^T Q\,\mathbf{x}$ | Encode in computational basis |
| **Ising** | $s_i \in \{-1,+1\}$ | $\sum J_{ij}s_is_j + \sum h_i s_i$ | Spin model on a graph |
| **Hamiltonian** | Qubits | $\hat{H} = \sum J_{ij}Z_iZ_j + \sum h_iZ_i$ | Ground state = optimal solution |

---
## Part III — Encoding Classical Problems as QUBO

Many NP-hard combinatorial problems have natural QUBO formulations:

- **Naturally unconstrained**: the objective is already a quadratic function of binary variables (e.g. Number Partitioning).
- **Constrained**: we add **penalty terms** $\lambda\,P(\mathbf{x})$ that vanish when constraints are satisfied (e.g. Vertex Cover, Graph Coloring).

In [ ]:
# ======== Number Partitioning =============================================
# Partition S = {3, 1, 4, 2} into two subsets with equal sum.
# QUBO: minimize (c - 2 * sum(w_i * x_i))^2
#   Q[i,i] = 4*w_i*(w_i - c),  Q[i,j] = 8*w_i*w_j  for i<j.

weights = np.array([3, 1, 4, 2], dtype=float)
n_part = len(weights)
c_sum = float(weights.sum())

Q_partition = np.zeros((n_part, n_part))
for i in range(n_part):
    Q_partition[i, i] = 4 * weights[i] * (weights[i] - c_sum)
    for j in range(i + 1, n_part):
        Q_partition[i, j] = 8 * weights[i] * weights[j]

print(f"Number Partitioning: S = {{{', '.join(str(int(w)) for w in weights)}}}")
print(f"Total sum c = {int(c_sum)}")
print(f"\nQ =")
print(Q_partition)

best_x_part, best_cost_part, all_part = brute_force_qubo(Q_partition)

print(f"\n  {'x':<12s}  {'QUBO cost':>10s}  {'diff\u00b2':>8s}")
print("  " + "-" * 36)
for k, x, cost in all_part:
    bits = "".join(str(int(b)) for b in x)
    diff_sq = cost + c_sum**2
    marker = "  \u2705" if abs(cost - best_cost_part) < 1e-9 else ""
    print(f"  {bits:<12s}  {cost:10.1f}  {diff_sq:8.1f}{marker}")

subset_1 = [int(weights[i]) for i in range(n_part) if best_x_part[i] == 1]
subset_0 = [int(weights[i]) for i in range(n_part) if best_x_part[i] == 0]
print(f"\nOptimal partition: {subset_0} | {subset_1}")
print(f"Sums: {sum(subset_0)} | {sum(subset_1)}")

fig, ax = plt.subplots(figsize=(12, 4))
plot_qubo_landscape(Q_partition, title="Number Partitioning QUBO Landscape", ax=ax)
plt.tight_layout()
plt.show()

assert abs(best_cost_part - (-c_sum**2)) < 1e-9, "Perfect partition should give cost -c\u00b2"
assert sum(subset_0) == sum(subset_1), "Partition sums must be equal!"
print(f"\n\u2705 Number Partitioning: perfect partition found (diff\u00b2 = 0)")

In [ ]:
# ======== Minimum Vertex Cover ============================================
# 4-node cycle C4.  Edges: (0,1),(1,2),(2,3),(0,3).
# Objective: minimize sum(x_i)  (cover size)
# Penalty: lambda*(1-x_i)(1-x_j) for each edge

n_vc = 4
edges_vc = [(0, 1), (1, 2), (2, 3), (0, 3)]
LAMBDA_VC = 10

Q_vc = np.zeros((n_vc, n_vc))
for i in range(n_vc):
    Q_vc[i, i] += 1.0
for i, j in edges_vc:
    Q_vc[i, i] -= LAMBDA_VC
    Q_vc[j, j] -= LAMBDA_VC
    ii, jj = min(i, j), max(i, j)
    Q_vc[ii, jj] += LAMBDA_VC

best_x_vc, best_cost_vc, _ = brute_force_qubo(Q_vc)

cover = [i for i in range(n_vc) if best_x_vc[i] == 1]
cover_size = len(cover)
all_covered = all(best_x_vc[i] == 1 or best_x_vc[j] == 1 for i, j in edges_vc)

print(f"Vertex Cover on C\u2084 (\u03bb={LAMBDA_VC})")
print(f"Q =")
print(Q_vc)
print(f"\nOptimal cover: vertices {cover}  (size {cover_size})")
print(f"All edges covered: {all_covered}")

fig, ax = plt.subplots(figsize=(12, 4))
plot_qubo_landscape(Q_vc, title=f"Vertex Cover QUBO (\u03bb={LAMBDA_VC})", ax=ax)
plt.tight_layout()
plt.show()

assert all_covered, "Solution must cover all edges!"
assert cover_size == 2, f"Min cover of C4 has size 2, got {cover_size}"
print(f"\n\u2705 Minimum Vertex Cover: valid cover of size {cover_size}")

In [ ]:
# ======== Graph 2-Coloring ================================================
# Bipartite graph K_{2,2}: partition {0,1} and {2,3}.
# Penalty for same-color adjacent: 2*x_i*x_j - x_i - x_j + 1 per edge

n_gc = 4
edges_gc = [(0, 2), (0, 3), (1, 2), (1, 3)]

Q_gc = np.zeros((n_gc, n_gc))
const_gc = 0
for i, j in edges_gc:
    Q_gc[i, i] -= 1.0
    Q_gc[j, j] -= 1.0
    ii, jj = min(i, j), max(i, j)
    Q_gc[ii, jj] += 2.0
    const_gc += 1

best_x_gc, best_cost_gc, _ = brute_force_qubo(Q_gc)
penalty_val = best_cost_gc + const_gc
coloring = [int(best_x_gc[i]) for i in range(n_gc)]
violations = sum(1 for i, j in edges_gc if best_x_gc[i] == best_x_gc[j])

print("2-Coloring of K_{2,2} (bipartite graph)")
print(f"Edges: {edges_gc}")
print(f"Q =")
print(Q_gc)
print(f"\nOptimal coloring: {coloring}")
print(f"Same-color-edge penalty: {int(penalty_val)}")
print(f"Violations: {violations}")

fig, ax = plt.subplots(figsize=(12, 4))
plot_qubo_landscape(Q_gc, title="Graph 2-Coloring QUBO (K_{2,2})", ax=ax)
plt.tight_layout()
plt.show()

assert penalty_val < 1e-9, f"Bipartite graph must be 2-colorable, penalty={penalty_val}"
assert violations == 0, f"Expected 0 violations, got {violations}"
print("\n\u2705 Graph 2-Coloring: K_{2,2} is bipartite, zero penalty achieved")

### Encoding recipes

| Problem | Variables | Objective | Constraints | QUBO construction |
|---------|-----------|-----------|-------------|-------------------|
| **Number Partitioning** | $x_i=1$: item $i$ in subset A | $\min\bigl(\sum w_i - 2\sum w_i x_i\bigr)^2$ | None (unconstrained) | $Q_{ii}=4w_i(w_i-c)$, $Q_{ij}=8w_iw_j$ |
| **Vertex Cover** | $x_i=1$: vertex $i$ in cover | $\min \sum x_i$ | $x_i+x_j \ge 1\;\forall (i,j)\in E$ | Add $\lambda(1-x_i)(1-x_j)$ per edge |
| **Graph 2-Coloring** | $x_i \in \{0,1\}$: color | $\min$ same-color penalties | None (penalty *is* objective) | Add $2x_ix_j - x_i - x_j + 1$ per edge |

---
## Part IV — Constraint Handling with Penalty Functions

For constrained problems, we convert to QUBO via penalties:

$$\min_{\mathbf{x}} \; f(\mathbf{x}) \;+\; \lambda\, P(\mathbf{x})$$

where $P(\mathbf{x}) = 0$ if and only if all constraints are satisfied.

| Constraint type | Penalty form |
|-----------------|-------------|
| Equality $g(\mathbf{x})=0$ | $\lambda\, g(\mathbf{x})^2$ |
| Inequality $g(\mathbf{x}) \le 0$ | Introduce slack variables |

**Choosing $\lambda$:** must be large enough that violating any constraint always costs more than the maximum feasible objective difference. Too small $\to$ constraint violations; too large $\to$ numerical issues.

In [ ]:
# ======== Penalty strength analysis =======================================
# Sweep lambda for the vertex-cover problem on C4.

lambdas = np.concatenate([np.arange(0.1, 2.05, 0.1),
                          np.arange(2.0, 21.0, 1.0)])
feasible_flags = []
cover_sizes_sweep = []
opt_costs_sweep = []

for lam in lambdas:
    Qt = np.zeros((n_vc, n_vc))
    for i in range(n_vc):
        Qt[i, i] += 1.0
    for i, j in edges_vc:
        Qt[i, i] -= lam
        Qt[j, j] -= lam
        ii, jj = min(i, j), max(i, j)
        Qt[ii, jj] += lam
    bx, bc, _ = brute_force_qubo(Qt)
    csize = int(bx.sum())
    feasible = all(bx[i] == 1 or bx[j] == 1 for i, j in edges_vc)
    feasible_flags.append(feasible)
    cover_sizes_sweep.append(csize)
    opt_costs_sweep.append(bc)

threshold_idx = next((i for i, f in enumerate(feasible_flags) if f), len(lambdas))
threshold_lam = lambdas[threshold_idx] if threshold_idx < len(lambdas) else None

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_f = ["#4CAF50" if f else "#F44336" for f in feasible_flags]
ax1.scatter(lambdas, cover_sizes_sweep, c=colors_f, edgecolor="black",
            linewidth=0.5, s=50)
if threshold_lam is not None:
    ax1.axvline(x=threshold_lam, color="orange", linestyle="--", linewidth=2,
              label=f"\u03bb* = {threshold_lam:.1f}")
ax1.set_xlabel("\u03bb (penalty strength)")
ax1.set_ylabel("Cover size")
ax1.set_title("Feasibility vs \u03bb")
ax1.legend()
ax1.grid(axis="y", alpha=0.25)

ax2.plot(lambdas, opt_costs_sweep, "o-", color="#2196F3", markersize=4)
if threshold_lam is not None:
    ax2.axvline(x=threshold_lam, color="orange", linestyle="--", linewidth=2,
              label=f"\u03bb* = {threshold_lam:.1f}")
ax2.set_xlabel("\u03bb (penalty strength)")
ax2.set_ylabel("Optimal QUBO cost")
ax2.set_title("QUBO cost vs \u03bb")
ax2.legend()
ax2.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

assert feasible_flags[-1], "At \u03bb=20, solution must be feasible!"
assert cover_sizes_sweep[-1] == 2, f"At \u03bb=20, min cover should be 2, got {cover_sizes_sweep[-1]}"
print(f"Threshold \u03bb* = {threshold_lam:.1f}")
print(f"\n\u2705 Penalty analysis: feasible solutions found for \u03bb \u2265 {threshold_lam:.1f}")

### Practical guidelines for penalty selection

- **Rule of thumb:** $\lambda > \max_{\text{feasible}}|f(\mathbf{x}_1) - f(\mathbf{x}_2)|$, i.e. larger than the biggest objective gap among feasible solutions.
- **Too small:** optimizer prefers constraint-violating solutions with lower apparent cost.
- **Too large:** all infeasible states are pushed far away, flattening the landscape among feasible states and causing numerical issues.
- **Connection to classical optimization:** the penalty approach mirrors Lagrangian relaxation; $\lambda$ plays the role of a Lagrange multiplier.

---
## Part V — Solving QUBO with QAOA

The full pipeline:

$$\text{QUBO} \;\xrightarrow{\text{convert}}\; \text{Ising } (J, h) \;\xrightarrow{\text{encode}}\; \hat{H}_C \;\xrightarrow{\text{QAOA}}\; |\boldsymbol{\gamma},\boldsymbol{\beta}\rangle \;\xrightarrow{\text{measure}}\; \mathbf{x}^*$$

1. Convert the QUBO matrix $Q$ to Ising parameters $(J, h)$.
2. Build the QAOA ansatz: $|\boldsymbol{\gamma},\boldsymbol{\beta}\rangle = \prod_{l=1}^p e^{-i\beta_l \hat{B}}\,e^{-i\gamma_l \hat{H}_C}\,|{+}\rangle^{\otimes n}$
3. Optimize $(\boldsymbol{\gamma}, \boldsymbol{\beta})$ classically to minimize $\langle \hat{H}_C \rangle$.
4. Measure the final state to obtain a candidate binary solution.

In [ ]:
# ======== QAOA solver for QUBO ============================================

def build_qaoa_circuit_ising(n, J, h, gammas, betas, measure=False):
    """Build a QAOA circuit for a general Ising cost Hamiltonian."""
    p = len(gammas)
    assert len(betas) == p
    qc = QuantumCircuit(n, n) if measure else QuantumCircuit(n)
    for i in range(n):
        qc.h(i)
    for layer in range(p):
        qc.barrier()
        # Cost unitary
        for i in range(n):
            for j in range(i + 1, n):
                if abs(J[i, j]) > 1e-12:
                    qc.cx(i, j)
                    qc.rz(2 * gammas[layer] * J[i, j], j)
                    qc.cx(i, j)
        for i in range(n):
            if abs(h[i]) > 1e-12:
                qc.rz(2 * gammas[layer] * h[i], i)
        qc.barrier()
        # Mixer unitary
        for i in range(n):
            qc.rx(2 * betas[layer], i)
    if measure:
        qc.barrier()
        qc.measure(range(n), range(n))
    return qc

# Convert Number Partitioning QUBO to Ising
J_part, h_part, offset_part = qubo_to_ising(Q_partition)

# Precompute QUBO costs for fast expectation evaluation
all_qubo_costs = np.array([
    qubo_cost(np.array([(k >> i) & 1 for i in range(n_part)], dtype=float),
             Q_partition)
    for k in range(2**n_part)
])
avg_qubo_cost = float(all_qubo_costs.mean())

def qaoa_objective(params, p):
    """Expected QUBO cost for given QAOA parameters."""
    qc = build_qaoa_circuit_ising(n_part, J_part, h_part,
                                  params[:p], params[p:])
    sv = Statevector.from_instruction(qc)
    return float(sv.probabilities() @ all_qubo_costs)

# Optimize at p=1 and p=2
results_qaoa = {}
rng = np.random.RandomState(SEED)

for p_val in [1, 2]:
    best_c, best_p = np.inf, None
    for trial in range(10):
        x0 = rng.uniform(-np.pi, np.pi, 2 * p_val)
        res = minimize(qaoa_objective, x0, args=(p_val,),
                       method="COBYLA", options={"maxiter": 500})
        if res.fun < best_c:
            best_c, best_p = res.fun, res.x.copy()
    results_qaoa[p_val] = {"cost": best_c, "params": best_p}
    print(f"QAOA p={p_val}: expected QUBO cost = {best_c:.2f}")

print(f"\nBrute-force optimal: {best_cost_part:.1f}")
print(f"Random (uniform) avg: {avg_qubo_cost:.1f}")

assert results_qaoa[1]["cost"] < avg_qubo_cost, \
    f"QAOA p=1 must beat random avg {avg_qubo_cost:.1f}, got {results_qaoa[1]['cost']:.2f}"
assert results_qaoa[2]["cost"] <= results_qaoa[1]["cost"] + 5, \
    "QAOA p=2 should be comparable to or better than p=1"
print("\n\u2705 QAOA optimization successful")

In [ ]:
# ======== Visualize QAOA results ==========================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

for ax, p_val in [(ax1, 1), (ax2, 2)]:
    par = results_qaoa[p_val]["params"]
    qc = build_qaoa_circuit_ising(n_part, J_part, h_part,
                                  par[:p_val], par[p_val:])
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities()

    labels = ["".join(str((k >> i) & 1) for i in range(n_part))
              for k in range(2**n_part)]
    norm = mplNormalize(vmin=all_qubo_costs.min(), vmax=all_qubo_costs.max())
    colors_q = cm.viridis_r(norm(all_qubo_costs))

    ax.bar(range(2**n_part), probs, color=colors_q,
           edgecolor="black", linewidth=0.5)
    ax.set_xticks(range(2**n_part))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_ylabel("Probability")
    ax.set_title(f"QAOA p={p_val}  (cost = {results_qaoa[p_val]['cost']:.1f})")
    ax.grid(axis="y", alpha=0.25)

plt.tight_layout()
plt.show()

# Print top outcomes
for p_val in [1, 2]:
    par = results_qaoa[p_val]["params"]
    qc = build_qaoa_circuit_ising(n_part, J_part, h_part,
                                  par[:p_val], par[p_val:])
    sv = Statevector.from_instruction(qc)
    probs = sv.probabilities()
    top_k = np.argsort(probs)[::-1][:5]
    print(f"\nTop outcomes (p={p_val}):")
    for k in top_k:
        x = np.array([(k >> i) & 1 for i in range(n_part)], dtype=float)
        bits = "".join(str(int(b)) for b in x)
        diff_sq = all_qubo_costs[k] + c_sum**2
        print(f"  |{bits}\u27e9  prob={probs[k]:.4f}  QUBO={all_qubo_costs[k]:.0f}  diff\u00b2={diff_sq:.0f}")

print("\n\u2705 QAOA probability distributions plotted")

In [ ]:
# ======== Shot-based validation ===========================================

p_final = 2
par_final = results_qaoa[p_final]["params"]
qc_meas = build_qaoa_circuit_ising(n_part, J_part, h_part,
                                   par_final[:p_final], par_final[p_final:],
                                   measure=True)
compiled = transpile(qc_meas, backend, seed_transpiler=SEED)
result = backend.run(compiled, shots=SHOTS, seed_simulator=SEED).result()
counts = result.get_counts()

# Empirical expected cost
empirical_cost = sum(count * all_qubo_costs[int(bs, 2)]
                     for bs, count in counts.items()) / SHOTS

exact_cost = results_qaoa[p_final]["cost"]
diff_val = abs(exact_cost - empirical_cost)

print(f"Exact expected cost:    {exact_cost:.4f}")
print(f"Empirical ({SHOTS} shots): {empirical_cost:.4f}")
print(f"Difference:             {diff_val:.4f}")

# Top measured outcomes
sorted_counts = sorted(counts.items(), key=lambda item: item[1], reverse=True)
print(f"\nTop measured outcomes:")
for bs, count in sorted_counts[:5]:
    k = int(bs, 2)
    x = np.array([(k >> i) & 1 for i in range(n_part)], dtype=float)
    bits = "".join(str(int(b)) for b in x)
    diff_sq = all_qubo_costs[k] + c_sum**2
    print(f"  |{bits}\u27e9  freq={count/SHOTS:.3f}  QUBO={all_qubo_costs[k]:.0f}  diff\u00b2={diff_sq:.0f}")

assert diff_val < 15, \
    f"Empirical should match exact within 15, got diff={diff_val:.2f}"
print(f"\n\u2705 Shot-based simulation matches exact expectation (diff = {diff_val:.4f})")

---
## TODO — Advanced Topics

- **Higher-order binary optimization** (HUBO / PUBO): extend to cubic and higher-degree terms via auxiliary variables.
- **Quantum annealing and D-Wave connection**: QUBO is the native input format for D-Wave quantum annealers.
- **Real-world QUBO applications**: portfolio optimization, job-shop scheduling, vehicle routing.
- **Hybrid decomposition**: large QUBO instances solved via divide-and-conquer with quantum sub-solvers.
- **Warm-starting QAOA**: initialize variational parameters from classical heuristic solutions.

---
## Takeaways

| Concept | Key Insight |
|---------|-------------|
| **QUBO** | Minimize $\mathbf{x}^T Q\,\mathbf{x}$ over binary $\mathbf{x}$; universal framework for combinatorial optimization |
| **$Q$ matrix** | Diagonal = linear costs, off-diagonal = pairwise interactions between variables |
| **QUBO $\leftrightarrow$ Ising** | Substitution $x_i = (1-s_i)/2$ converts between binary and spin representations |
| **Ising $\to$ Hamiltonian** | Replace $s_i$ with $Z_i$ operator; ground state = optimal solution |
| **Number Partitioning** | Minimize squared subset-sum difference $\to$ natural QUBO formulation |
| **Vertex Cover** | Minimize cover size + penalty for uncovered edges |
| **Penalty method** | $f(\mathbf{x}) + \lambda\,P(\mathbf{x})$: $\lambda$ must exceed max feasible objective difference |
| **QUBO $\to$ QAOA pipeline** | QUBO $\to$ Ising $\to$ Hamiltonian $\to$ QAOA circuit $\to$ classical optimization |